In [1]:
# importing necessary libraries
import numpy as np
import ast
import time
import os
from sentence_transformers import SentenceTransformer
import pandas as pd
from ingredient_parser import parse_ingredient
from tqdm import tqdm
import json
import hdbscan
from collections import defaultdict


### Phase 1: Data Cleaning & Synonym Generation

#### Data Cleaning

We've taken following steps to clean the data.
1. **Parsed String-Lists**: Columns like ingredients were just useless strings (e.g., '["item 1", "item 2"]'). We converted them into actual Python lists (e.g., ["item 1", "item 2"]) that a program can read and search. We did this for ingredients, steps, and nutrition.

2. **Expanded Nutrition Data:** We took the nutrition list (e.g., [250, 10.5, ...]) and split it into its own set of new, separate columns: calories, total_fat_g, sugar_g, sodium_mg, protein_g, etc. This makes nutritional analysis much easier.

3. **Removed "Useless" Recipes:** We dropped any recipes that were missing critical information, specifically any that had no name, no ingredients, or no steps.

4. **Standardized Data Types:** We made sure numeric columns (like minutes) were actually stored as numbers, not text, so we can perform calculations on them.

5. **Filled Missing Descriptions:** We replaced any missing description values (which were NaN) with an empty string ('') to prevent errors in the future.

6. **Removed Unneeded Columns:** We dropped contributor_id and submitted because they aren't needed for our app.


In [2]:
# --- Configuration ---
INPUT_FILE = './Data/RAW_recipes.csv'

# Define the directory and filename
OUTPUT_DIR = './Data/'
OUTPUT_FILENAME = 'recipes_cleaned.parquet'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns that are stored as string-lists and need parsing
LIST_COLUMNS = ['nutrition', 'steps', 'ingredients']

# Columns for the nutrition list, in the order they appear
NUTRITION_COLUMNS = [
    'calories',
    'total_fat_g',
    'sugar_g',
    'sodium_mg',
    'protein_g',
    'saturated_fat_g',
    'carbohydrates_g'
]

In [3]:
# --- Helper Function ---
def safe_parse_list(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError, TypeError):
        return []

# --- Main Processing Workflow ---
print("Starting data cleaning process...")
start_time = time.time()

# 1. Load Data
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} recipes from {INPUT_FILE}")
except FileNotFoundError:
    print(f"ERROR: Input file not found at {INPUT_FILE}")
    exit()

# 2. Parse String-Literal Columns
print(f"Parsing string-literal columns: {LIST_COLUMNS}...")
for col in LIST_COLUMNS:
    df[col] = df[col].apply(safe_parse_list)

# 3. Handle Critical Missing Data (dropping recipies that are missing names, ingredients and steps)
df.dropna(subset=['name', 'ingredients', 'steps'], inplace=True)
df = df[df['ingredients'].str.len() > 0]
df = df[df['steps'].str.len() > 0]
print(f"Recipes after dropping critical missing data: {len(df)}")

# 4. Expand the 'nutrition' Column
print("Expanding 'nutrition' column into separate columns...")
nutrition_df = pd.DataFrame(
    df['nutrition'].tolist(),
    index=df.index,
    columns=NUTRITION_COLUMNS
)
for col in NUTRITION_COLUMNS:
    nutrition_df[col] = pd.to_numeric(nutrition_df[col], errors='coerce')

df = df.join(nutrition_df)
df.drop(columns=['nutrition'], inplace=True)

# 5. Filled Missing Descriptions:
print("Cleaning up other columns...")
df['description'] = df['description'].fillna('')

# Standardized Data Types for minutes, number of steps and ingredients column
df['minutes'] = pd.to_numeric(df['minutes'], errors='coerce')
df['n_steps'] = pd.to_numeric(df['n_steps'], errors='coerce')
df['n_ingredients'] = pd.to_numeric(df['n_ingredients'], errors='coerce')

# 6. Removed Unneeded Columns as they are not needed for analysis
df.drop(columns=['contributor_id', 'submitted'], inplace=True, errors='ignore')

# Saving the cleaned data
print("Saving cleaned data to Parquet file...")
df.to_parquet(OUTPUT_FILE, index=False)

end_time = time.time()
print(f"---")
print(f"Success! Cleaning complete in {end_time - start_time:.2f} seconds.")
print(f"Final dataset has {len(df)} recipes.")
print(f"Cleaned data saved to {OUTPUT_FILE}")

Starting data cleaning process...
Loaded 231637 recipes from ./Data/RAW_recipes.csv
Parsing string-literal columns: ['nutrition', 'steps', 'ingredients']...
Recipes after dropping critical missing data: 231635
Expanding 'nutrition' column into separate columns...
Cleaning up other columns...
Saving cleaned data to Parquet file...
---
Success! Cleaning complete in 9.65 seconds.
Final dataset has 231635 recipes.
Cleaned data saved to ./Data/recipes_cleaned.parquet


#### Synonym Generation
This is what we just did. It's a one-time process to create our master data.

1. Parse Ingredients: Looping through all RAW_recipes.csv, use ingredient-parser-nlp to extract every unique ingredient name. We will get `unique_ingredients.json` (e.g., ["red pepper", "capsicum", ...])

2. Embed Ingredients: Using SentenceTransformer to convert each name from unique_ingredients.json into a vector (a numerical representation of its meaning). We will get `ingredient_names.json` & `ingredient_vectors.npy`

3. Cluster Ingredients: Using HDBSCAN to find groups of vectors that are close together to get `synonym_map.json` (This maps all synonyms to a single parent name, e.g., "red pepper" -> "bell pepper").

4. Final Normalization: Loading recipes_cleaned.parquet, and creating a new ingredients_normalized column, and uses the synonym map to fill it. This will give us `recipes_final.parquet` which will be our "gold" dataset.

In [5]:


# --- Configuration ---
CLEANED_DATA_FILE = './Data/recipes_cleaned.parquet'
OUTPUT_FILE = './Data/unique_ingredients.json'

# --- Main Script ---
print(f"Loading cleaned data from {CLEANED_DATA_FILE}...")
try:
    df = pd.read_parquet(CLEANED_DATA_FILE)
except FileNotFoundError:
    print(f"ERROR: File not found. Make sure this path is correct: {CLEANED_DATA_FILE}")
    exit()

print(f"Loaded {len(df)} recipes. Starting ingredient parsing...")
start_time = time.time()

# This set will store all unique ingredient names we find
unique_ingredient_names = set()
total_ingredients_parsed = 0

# Use tqdm for a progress bar as this will take a while
for ingredient_list in tqdm(df['ingredients'], desc="Parsing Recipes"):

    # Now, iterate through each ingredient string in that recipe's list
    for ingredient_string in ingredient_list:
        try:
            # This is where the magic happens
            parsed = parse_ingredient(ingredient_string)

            # The 'parsed' object has a '.name' attribute
            # This attribute is a LIST of name parts. We just want the main one.
            if parsed and parsed.name:
                # Get the text of the first (most likely) name part
                name = parsed.name[0].text

                if name:
                    # Add the cleaned, lowercase name to our set
                    unique_ingredient_names.add(name.lower())
                    total_ingredients_parsed += 1

        except Exception as e:
            # Catch any weird parsing errors
            # print(f"Could not parse '{ingredient_string}': {e}")
            pass

end_time = time.time()
print(f"\n--- Processing Complete ---")
print(f"Finished in {end_time - start_time:.2f} seconds.")
print(f"Parsed a total of {total_ingredients_parsed:,} ingredient strings.")
print(f"Found {len(unique_ingredient_names):,} unique ingredient names.")

# Convert the set to a list so it can be saved as JSON
unique_list = list(unique_ingredient_names)

print(f"Saving unique names to {OUTPUT_FILE}...")
with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
    json.dump(unique_list, f, indent=2)

print("Done.")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/verdabatool/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Loading cleaned data from ./Data/recipes_cleaned.parquet...
Loaded 231635 recipes. Starting ingredient parsing...


Parsing Recipes: 100%|██████████| 231635/231635 [06:30<00:00, 593.52it/s]


--- Processing Complete ---
Finished in 390.28 seconds.
Parsed a total of 2,096,464 ingredient strings.
Found 14,242 unique ingredient names.
Saving unique names to unique_ingredients.json...
Done.


In [8]:
# --- Configuration ---
INPUT_FILE = './Data/unique_ingredients.json'
MODEL_NAME = 'all-MiniLM-L6-v2' # A fast, high-quality model
OUTPUT_NAMES_FILE = './Data/ingredient_names.json'
OUTPUT_VECTORS_FILE = './Data/ingredient_vectors.npy'

# --- Main Script ---
print(f"Loading unique ingredient names from {INPUT_FILE}...")
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    names = json.load(f)

print(f"Loaded {len(names):,} unique names.")
print(f"Loading model '{MODEL_NAME}'. This may take a moment...")
start_time = time.time()

# This will download the model the first time you run it
# It will try to use your Mac's GPU/NPU automatically
model = SentenceTransformer(MODEL_NAME)

print(f"Model loaded in {time.time() - start_time:.2f} seconds.")
print("Starting to encode names into vectors. This is the slow part...")

start_time = time.time()

# This one line does all the work.
# It processes the names in batches for high efficiency.
vectors = model.encode(names, show_progress_bar=True)

end_time = time.time()
print(f"\n--- Encoding Complete ---")
print(f"Finished in {end_time - start_time:.2f} seconds.")
print(f"Created {len(vectors)} vectors, each with {vectors.shape[1]} dimensions.")

# --- Save the results ---
print(f"Saving vectors to {OUTPUT_VECTORS_FILE}...")
# We save the vectors as a highly efficient NumPy binary file
np.save(OUTPUT_VECTORS_FILE, vectors)

print(f"Saving corresponding names list to {OUTPUT_NAMES_FILE}...")
# We also save the names list *in the same order*
# This is CRITICAL so we know which vector maps to which name
with open(OUTPUT_NAMES_FILE, 'w', encoding='utf-8') as f:
    json.dump(names, f)

print("Done.")
print("\nNext step: Clustering these vectors.")

Loading unique ingredient names from ./Data/unique_ingredients.json...
Loaded 14,242 unique names.
Loading model 'all-MiniLM-L6-v2'. This may take a moment...
Model loaded in 7.79 seconds.
Starting to encode names into vectors. This is the slow part...


Batches:   0%|          | 0/446 [00:00<?, ?it/s]


--- Encoding Complete ---
Finished in 3.24 seconds.
Created 14242 vectors, each with 384 dimensions.
Saving vectors to ./Data/ingredient_vectors.npy...
Saving corresponding names list to ./Data/ingredient_names.json...
Done.

Next step: Clustering these vectors.


In [9]:
# --- Configuration ---
NAMES_FILE = './Data/ingredient_names.json'
VECTORS_FILE = './Data/ingredient_vectors.npy'
OUTPUT_MAP_FILE = './Data/synonym_map.json'

# --- Main Script ---
print(f"Loading names from {NAMES_FILE}...")
with open(NAMES_FILE, 'r', encoding='utf-8') as f:
    names = json.load(f)

print(f"Loading vectors from {VECTORS_FILE}...")
vectors = np.load(VECTORS_FILE)

if len(names) != len(vectors):
    print("Error: Names and vectors files have different lengths. Aborting.")
    exit()

print(f"Loaded {len(names)} names and vectors.")
print("Starting clustering... This may take several minutes.")
start_time = time.time()

# Configure HDBSCAN
# min_cluster_size=2: We are looking for any group, even just two synonyms.
# metric='euclidean': Standard distance metric
clusterer = hdbscan.HDBSCAN(
    min_cluster_size=2,
    metric='euclidean',
    cluster_selection_epsilon=0.5 # Adjust this value (0.1 to 1.0) if you get too many/too few clusters
)

# This is the main processing step
cluster_labels = clusterer.fit_predict(vectors)

end_time = time.time()
print(f"Clustering complete in {end_time - start_time:.2f} seconds.")

# --- Process the clusters ---
num_clusters = len(set(cluster_labels)) - (1 if -1 in cluster_labels else 0)
num_noise = list(cluster_labels).count(-1)
print(f"Found {num_clusters} potential synonym clusters.")
print(f"Found {num_noise} 'noise' points (ingredients with no synonyms).")

# We'll use a defaultdict to group names by their cluster label
clusters = defaultdict(list)
for i, label in enumerate(cluster_labels):
    if label != -1:  # -1 means it's a "noise" point
        clusters[label].append(names[i])

# --- Create the final synonym map ---
synonym_map = {}
for cluster_id, items in clusters.items():
    if len(items) > 1:
        # Strategy: Pick the shortest item in the cluster as the "parent"
        parent_name = min(items, key=len)

        # Add all items in the cluster to the map, mapping to the parent
        for item in items:
            synonym_map[item] = parent_name

print(f"Created a synonym map with {len(synonym_map)} entries.")

# Save the map
print(f"Saving synonym map to {OUTPUT_MAP_FILE}...")
with open(OUTPUT_MAP_FILE, 'w', encoding='utf-8') as f:
    json.dump(synonym_map, f, indent=2)

print("Done.")

Loading names from ./Data/ingredient_names.json...
Loading vectors from ./Data/ingredient_vectors.npy...
Loaded 14242 names and vectors.
Starting clustering... This may take several minutes.


/Users/verdabatool/MS AI/natural language processing/NutriBot/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(
/Users/verdabatool/MS AI/natural language processing/NutriBot/.venv/lib/python3.11/site-packages/sklearn/utils/deprecation.py:132: FutureWarning: 'force_all_finite' was renamed to 'ensure_all_finite' in 1.6 and will be removed in 1.8.
  warnings.warn(


Clustering complete in 48.15 seconds.
Found 1153 potential synonym clusters.
Found 7280 'noise' points (ingredients with no synonyms).
Created a synonym map with 6962 entries.
Saving synonym map to ./Data/synonym_map.json...
Done.


In [10]:
# --- Configuration ---
# The last file we cleaned (before normalization)
CLEANED_FILE = './Data/recipes_cleaned.parquet'
# The map we just created with the clustering script
MAP_FILE = './Data/synonym_map.json'
# The final, "gold" dataset we are creating
OUTPUT_FILE = './Data/recipes_final.parquet'

# --- Main Script ---
print("--- Starting Final Normalization ---")

# 1. Load Inputs
print(f"Loading {CLEANED_FILE}...")
if not os.path.exists(CLEANED_FILE):
    print(f"ERROR: File not found: {CLEANED_FILE}")
    exit()
df = pd.read_parquet(CLEANED_FILE)

print(f"Loading synonym map from {MAP_FILE}...")
if not os.path.exists(MAP_FILE):
    print(f"ERROR: File not found: {MAP_FILE}")
    exit()
with open(MAP_FILE, 'r', encoding='utf-8') as f:
    synonym_map = json.load(f)

print(f"Loaded {len(df)} recipes and {len(synonym_map)} synonym rules.")
print("---")

# 2. Define the Normalization Function
# This function will be applied to every single recipe
def normalize_ingredient_list(original_list: list) -> list:
    """
    Takes a list of original ingredient strings (e.g., "1/2 cup red pepper").
    Returns a list of normalized parent names (e.g., "bell pepper").
    """
    normalized_set = set() # Use a set to avoid duplicates (e.g., "chicken breast" and "chicken")

    for ingredient_string in original_list:
        try:
            # Parse the string (e.g., "1/2 cup red pepper")
            parsed = parse_ingredient(ingredient_string)

            if parsed and parsed.name:
                # Get the parsed name (e.g., "red pepper")
                name = parsed.name[0].text.lower()

                # This is the core logic:
                # 1. Look up 'name' in our map.
                # 2. If it's found (like "red pepper"), use its value ("bell pepper").
                # 3. If it's not found, it's already a parent, so use 'name' itself.
                parent_name = synonym_map.get(name, name)

                normalized_set.add(parent_name)

        except Exception:
            # Ignore any strings the parser can't handle
            pass

    return list(normalized_set)

# 3. Apply the Function
print("Applying normalization to all recipes...")
print("(This is the slowest step and may take several minutes.)")
start_time = time.time()

# We need to tell tqdm to work with pandas .apply()
tqdm.pandas(desc="Normalizing Recipes")

# Create the new column by applying our function to the 'ingredients' column
df['ingredients_normalized'] = df['ingredients'].progress_apply(normalize_ingredient_list)

end_time = time.time()
print(f"Normalization complete in {end_time - start_time:.2f} seconds.")

# 4. Save the "Gold" Dataset
print(f"Saving final, normalized data to {OUTPUT_FILE}...")
df.to_parquet(OUTPUT_FILE, index=False)
print("---")

# 5. Verification
print(f"Success! {OUTPUT_FILE} is ready.")
print("\n--- Verification: Before and After ---")

# Show 5 random examples of the change
for _, row in df.sample(5).iterrows():
    print(f"**RECIPE:** {row['name']}")
    print(f"**Original:** {row['ingredients']}")
    print(f"**Normalized:** {row['ingredients_normalized']}")
    print("---")

--- Starting Final Normalization ---
Loading ./Data/recipes_cleaned.parquet...
Loading synonym map from ./Data/synonym_map.json...
Loaded 231635 recipes and 6962 synonym rules.
---
Applying normalization to all recipes...
(This is the slowest step and may take several minutes.)


Normalizing Recipes: 100%|██████████| 231635/231635 [06:22<00:00, 605.26it/s]


Normalization complete in 382.71 seconds.
Saving final, normalized data to ./Data/recipes_final.parquet...
---
Success! ./Data/recipes_final.parquet is ready.

--- Verification: Before and After ---
**RECIPE:** apricot glazed ham steaks
**Original:** ['lean ham' 'apricot fruit spread' 'orange juice']
**Normalized:** ['lean ham', 'apricot fruit spread', 'orange juice']
---
**RECIPE:** steamed mussels
**Original:** ['spaghetti sauce' 'white wine' 'fresh garlic' 'bay leaf' 'fresh mussels']
**Normalized:** ['pasta sauce', 'bay leaf', 'wine', 'mussels', 'garlic']
---
**RECIPE:** jacques pepin family style shrimp
**Original:** ['shrimp' 'onion' 'carrot' 'fresh thyme' 'bay leaves' 'lemon'
 'whole black peppercorn' 'red pepper flakes' 'salt' 'dry white wine'
 'water' 'butter' 'fresh lemon juice' 'black pepper']
**Normalized:** ['lemon', 'butter', 'shrimp', 'thyme', 'salt', 'black pepper', 'whole black peppercorn', 'bay leaf', 'chili flakes', 'wine', 'lemon juice', 'ice', 'onion', 'carrot']
---